# Section Summary: Data Enrichment & Joining

## Goal

In this notebook, we combine data from listings, reviews, and calendar datasets to create enriched datasets that support analytics and business intelligence.

## Objectives

- Join listings with review summaries.
- Calculate occupancy metrics.
- Create derived features.
- Prepare a master dataset for analysis.

In [1]:
import pandas as pd

In [2]:
# Load cleaned datasets

listings = pd.read_csv(
    "../data/processed/listings_cleaned.csv"
)

reviews = pd.read_csv(
    "../data/processed/reviews_cleaned.csv"
)

calendar = pd.read_csv(
    "../data/raw/calendar.csv"
)

In [3]:
# Section 3.3: Data Enrichment & Joining
# Task: Explore review data

print(reviews.columns.tolist())

['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']


In [4]:
# Section 3.3: Data Enrichment & Joining
# Task: Aggregate reviews by listing

review_summary = (
    reviews.groupby("listing_id")
    .agg(
        total_reviews=("id", "count"),
        first_review=("date", "min"),
        last_review=("date", "max")
    )
    .reset_index()
)

review_summary.head()

,listing_id,total_reviews,first_review,last_review
0,2539,10,2015-12-04,2026-05-28
1,6848,198,2009-05-25,2026-04-23
2,6872,2,2022-06-05,2025-10-07
3,6990,251,2009-10-28,2026-01-11
4,7097,423,2010-01-16,2025-09-23


In [5]:
# Section 3.3: Data Enrichment & Joining
# Task: Create an enriched listings table

enriched_listings = listings.merge(
    review_summary,
    left_on="id",
    right_on="listing_id",
    how="left"
)

enriched_listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,listing_id,total_reviews,first_review_y,last_review_y
0,2539,https://www.airbnb.com/rooms/2539,20260614073253,2026-06-15,city scrape,11 Min to Manhattan • Prospect Park • Fast WiFi,"Bright, serene room in a renovated apartment h...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,2787,...,NaN,5,0,5,0,0.08,2539.0,10.0,2015-12-04,2026-05-28
1,6848,https://www.airbnb.com/rooms/6848,20260614073253,2026-06-14,city scrape,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,https://a0.muscache.com/pictures/e4f031a7-f146...,15991,...,NaN,1,1,0,0,0.95,6848.0,198.0,2009-05-25,2026-04-23
2,6872,https://www.airbnb.com/rooms/6872,20260614073253,2026-06-14,city scrape,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,16104,...,NaN,2,0,2,0,0.04,6872.0,2.0,2022-06-05,2025-10-07
3,6990,https://www.airbnb.com/rooms/6990,20260614073253,2026-06-14,city scrape,UES Beautiful Blue Room,Beautiful peaceful healthy home,NaN,https://a0.muscache.com/pictures/45fb4ec7-6856...,16800,...,NaN,1,0,1,0,1.24,6990.0,251.0,2009-10-28,2026-01-11
4,7097,https://www.airbnb.com/rooms/7097,20260614073253,2026-06-14,city scrape,"Perfect for Your Parents, With Garden & Patio",Parents/grandparents coming to town or are you...,NaN,https://a0.muscache.com/pictures/aaac19fc-4b4d...,17571,...,NaN,2,0,2,0,2.12,7097.0,423.0,2010-01-16,2025-09-23


In [6]:
# Section 3.3: Data Enrichment & Joining
# Task: Validate the join

print(enriched_listings.shape)

(30259, 94)


In [7]:
# Section 3.3: Data Enrichment & Joining
# Task: Create derived features

enriched_listings["price_per_bedroom"] = (
    enriched_listings["price"] /
    enriched_listings["bedrooms"]
)

enriched_listings["days_since_last_review"] = (
    pd.Timestamp.today() -
    pd.to_datetime(enriched_listings["last_review_y"])
).dt.days

enriched_listings["listing_age_days"] = (
    pd.Timestamp.today() -
    pd.to_datetime(enriched_listings["first_review_y"])
).dt.days

In [8]:
# Section 3.3: Data Enrichment & Joining
# Task: Validate derived features

enriched_listings[
    [
        "price",
        "bedrooms",
        "price_per_bedroom",
        "first_review_y",
        "last_review_y",
        "listing_age_days",
        "days_since_last_review"
    ]
].head()

,price,bedrooms,price_per_bedroom,first_review_y,last_review_y,listing_age_days,days_since_last_review
0,113.97,1.0,113.970,2015-12-04,2026-05-28,3871.0,43.0
1,117.27,2.0,58.635,2009-05-25,2026-04-23,6255.0,78.0
2,80.06,NaN,NaN,2022-06-05,2025-10-07,1496.0,276.0
3,77.17,NaN,NaN,2009-10-28,2026-01-11,6099.0,180.0
4,202.47,NaN,NaN,2010-01-16,2025-09-23,6019.0,290.0


In [9]:
# Section 3.3: Data Enrichment & Joining
# Task: Save enriched listings

enriched_listings.to_csv(
    "../data/processed/enriched_listings.csv",
    index=False
)

# Section Summary: Data Enrichment & Joining

## Completed Tasks

- Aggregated review data by listing.
- Joined review summaries with listing information.
- Created an enriched listings dataset.
- Derived additional features such as price per bedroom and review-based metrics.

## Key Findings

- The enriched dataset contains 94 columns.
- Review history varies significantly between listings.
- Derived metrics provide additional business context for future analysis.

## Next Steps

The enriched dataset will be used for exploratory data analysis and statistical modeling.